In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load datasets
train = pd.read_csv("../data/train.csv")
calendar = pd.read_csv("../data/calendar_events.csv")
submission_template = pd.read_csv("../data/forecast_submission.csv")

LOOKBACK_DAYS = 28

In [2]:
from sklearn.linear_model import LinearRegression

LOOKBACK_DAYS = 28

trend_per_store = {}

for store_id, df in train.groupby("store_id"):
    df = df.sort_values("date").tail(LOOKBACK_DAYS)
    
    # Time index: 0,1,2,... (days)
    X = np.arange(len(df)).reshape(-1, 1)
    y = df["revenue"].values
    
    model = LinearRegression()
    model.fit(X, y)
    
    trend_per_store[store_id] = model.coef_[0]  # daily slope


In [3]:
list(trend_per_store.items())[:5]


[(0, np.float64(-1359.6974384236455)),
 (1, np.float64(-192.3552545155994)),
 (2, np.float64(-93.294540229885)),
 (3, np.float64(-155.00780788177337)),
 (4, np.float64(30.932933771209623))]

In [4]:
# Start from the weekday-based submission logic
sub = submission_template.copy()
sub["store_id"] = sub["id"].str.split("_").str[0].astype(int)
sub["date_str"] = sub["id"].str.split("_").str[1]
sub["date"] = pd.to_datetime(sub["date_str"], format="%Y%m%d")
sub["weekday"] = sub["date"].dt.weekday


In [8]:
train["date"] = pd.to_datetime(train["date"])
last_date = train["date"].max()
sub["days_ahead"] = (sub["date"] - last_date).dt.days


In [11]:
# make sure train date is datetime
train["date"] = pd.to_datetime(train["date"])

LOOKBACK_DAYS = 28

recent = (
    train.sort_values("date")
         .groupby("store_id")
         .tail(LOOKBACK_DAYS)
         .copy()
)

recent["weekday"] = recent["date"].dt.weekday

store_weekday_mean = recent.groupby(["store_id", "weekday"])["revenue"].mean()
store_mean = recent.groupby("store_id")["revenue"].mean()

store_weekday_mean.head(), store_mean.head()


(store_id  weekday
 0         0          291526.8000
           1          258842.7875
           2          254890.9200
           3          264703.6125
           4          293333.5425
 Name: revenue, dtype: float64,
 store_id
 0    298830.560714
 1     35568.902143
 2     32421.249643
 3     49842.347500
 4     19691.897143
 Name: revenue, dtype: float64)

In [12]:
# Base: weekday mean
sub["prediction"] = sub.set_index(["store_id", "weekday"]).index.map(store_weekday_mean)

# Fallback
missing = sub["prediction"].isna()
sub.loc[missing, "prediction"] = sub.loc[missing, "store_id"].map(store_mean)

# Add trend adjustment
sub["trend"] = sub["store_id"].map(trend_per_store)
sub["prediction"] += sub["trend"] * sub["days_ahead"]


In [13]:
sub[["id", "prediction"]].head()

,id,prediction
0,0_20151001,263343.915062
1,0_20151002,290614.147623
2,0_20151003,357592.182685
3,0_20151004,361406.197746
4,0_20151005,284728.312808


In [14]:
trend_submission = sub[["id", "prediction"]]

print("Rows:", len(trend_submission))
print("Any NaNs?", trend_submission["prediction"].isna().any())

trend_submission.to_csv("../submissions/weekday_trend_submission.csv", index=False)
print("Saved weekday_trend_submission.csv")


Rows: 1012
Any NaNs? False
Saved weekday_trend_submission.csv
